# 03 — E5_team / E7_team / (optional) Fusion on the team split

**Run in Google Colab on a T4 GPU.** Locally CLIP+CatBoost works, but on a Mac CPU the e5-small text encoding (Stage 2.5) takes ~hours; T4 does it in ~25 min.

## Before running

1. **Runtime → Change runtime type → T4 GPU → Save.**
2. Mount Drive (cell below) and place these in `/content/drive/MyDrive/team_runs/`:
   - `data/ozon_train.csv`  (190 MB)
   - `embeddings/clip_embeddings.parquet`  (591 MB — your local `/Users/sofya/Desktop/diplomahse/clip_embeddings.parquet`)
   - `splits/team_train_idx.npy`, `team_val_idx.npy`, `team_test_idx.npy`, `y_test.npy`  (from local Stage 1)
   - `notebooks/utils.py`  (from local Stage 1)
3. After running, download `proba/test_proba_e5_team.npy`, `test_proba_e7_team.npy`, optionally `test_proba_fusion_team.npy` and `embeddings/text_e5_small.parquet` back to your local `team_runs/`.

## What it does

1. (Stage 2.5) Generates `intfloat/multilingual-e5-small` text embeddings on **all** ItemIDs once and saves them to `embeddings/text_e5_small.parquet`. Skipped automatically on subsequent runs.
2. (Stage 3) Runs E5_team (CLIP+PCA-25), E7_team (text+PCA-25), and an optional Fusion (CLIP-PCA+text-PCA), with the same CatBoost config as Stage 2 (`E0_team_*`).

In [ ]:
# ---- Setup ---------------------------------------------------------------
import os, sys, json, time
import numpy as np
import pandas as pd

from google.colab import drive
drive.mount('/content/drive')

import torch
print(f'CUDA available: {torch.cuda.is_available()}, device: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"}')
if not torch.cuda.is_available():
    print('WARN: no GPU. Runtime > Change runtime type > T4 GPU.')

!pip install -q sentence-transformers catboost

TR     = '/content/drive/MyDrive/team_runs'
DATA   = f'{TR}/data/ozon_train.csv'
CLIP   = f'{TR}/embeddings/clip_embeddings.parquet'
SPLITS = f'{TR}/splits'
PROBA  = f'{TR}/proba'
EMB    = f'{TR}/embeddings'
os.makedirs(PROBA, exist_ok=True)
os.makedirs(EMB, exist_ok=True)

for p in [DATA, CLIP, f'{SPLITS}/team_train_idx.npy', f'{TR}/notebooks/utils.py']:
    assert os.path.exists(p), f'MISSING: {p}'

sys.path.insert(0, f'{TR}/notebooks')
from utils import normalize_emb_df, evaluate, r_at_p90, avg_precision_metric

In [ ]:
# ---- Load data + split + labels ------------------------------------------
df = pd.read_csv(DATA, encoding='utf-8')
df['ItemID'] = df['ItemID'].astype('int64')

train_idx = np.load(f'{SPLITS}/team_train_idx.npy')
val_idx   = np.load(f'{SPLITS}/team_val_idx.npy')
test_idx  = np.load(f'{SPLITS}/team_test_idx.npy')
y_test_saved = np.load(f'{SPLITS}/y_test.npy')

y_train = df.iloc[train_idx]['resolution'].astype('int8').values
y_val   = df.iloc[val_idx]['resolution'].astype('int8').values
y_test  = df.iloc[test_idx]['resolution'].astype('int8').values
assert (y_test == y_test_saved).all()

spw = float((y_train == 0).sum() / max((y_train == 1).sum(), 1))

CATBOOST_BASE = dict(
    iterations=1000, depth=6, learning_rate=0.05,
    eval_metric='AUC', scale_pos_weight=spw,
    early_stopping_rounds=50, random_seed=42, verbose=100,
)
print(f'sizes: train={len(train_idx)}, val={len(val_idx)}, test={len(test_idx)}')
print(f'scale_pos_weight = {spw:.4f}')

In [ ]:
# ---- Stage 2.5: e5-small text embeddings (one-shot, then cached) ---------
from sentence_transformers import SentenceTransformer

E5_PATH = f'{EMB}/text_e5_small.parquet'
TEXT_COLS = ['name_rus', 'description', 'brand_name', 'CommercialTypeName4']

if os.path.exists(E5_PATH):
    print(f'cached e5 embeddings exist, loading from {E5_PATH}')
    text_emb = pd.read_parquet(E5_PATH)
else:
    model = SentenceTransformer('intfloat/multilingual-e5-small')
    print(f'model loaded on {model.device}')

    def build_text(row):
        parts = []
        for col in TEXT_COLS:
            v = row.get(col)
            if pd.notna(v) and str(v).strip() and str(v) != 'nan':
                parts.append(str(v))
        return 'passage: ' + ' '.join(parts) if parts else 'passage: empty'

    print(f'building texts for {len(df)} rows...')
    texts = df[TEXT_COLS].apply(build_text, axis=1).tolist()
    t0 = time.time()
    embs = model.encode(texts, batch_size=128, show_progress_bar=True,
                        convert_to_numpy=True, normalize_embeddings=True)
    print(f'encoded in {time.time()-t0:.1f}s, shape={embs.shape}, dtype={embs.dtype}')

    text_emb = pd.DataFrame(embs.astype('float32'),
                            columns=[f't_{i}' for i in range(embs.shape[1])])
    text_emb.insert(0, 'ItemID', df['ItemID'].values)
    text_emb.to_parquet(E5_PATH, index=False)
    print(f'saved -> {E5_PATH} ({os.path.getsize(E5_PATH)/1e6:.1f} MB)')
    del model, texts, embs

assert text_emb.shape[1] - 1 == 384, f'expected 384 dim, got {text_emb.shape[1]-1}'
print(f'text_emb: {text_emb.shape}, ItemID dtype={text_emb["ItemID"].dtype}')

In [ ]:
# ---- Load + normalize CLIP -----------------------------------------------
clip_raw = pd.read_parquet(CLIP)
clip_norm = normalize_emb_df(clip_raw, expected_dim=512, prefix='clip', id_col='ItemID')
print(f'CLIP normalized: {clip_norm.shape}, ItemID dtype={clip_norm["ItemID"].dtype}')
del clip_raw

In [ ]:
# ---- Multimodal runner ----------------------------------------------------
from sklearn.decomposition import PCA
from catboost import CatBoostClassifier

MY_FEATURES = [c for c in df.columns if c not in ['resolution', 'ItemID', 'SellerID']]
MY_CATS = df[MY_FEATURES].select_dtypes(include=['object', 'string', 'category']).columns.tolist()
MY_NUMS = [c for c in MY_FEATURES if c not in MY_CATS]
print(f'MY_FEATURES={len(MY_FEATURES)}, MY_CATS={MY_CATS}')


def prep_tabular(idx):
    X = df.iloc[idx][MY_FEATURES].copy()
    if MY_NUMS:
        X[MY_NUMS] = X[MY_NUMS].fillna(0)
    for c in MY_CATS:
        X[c] = X[c].fillna('nan').astype(str)
    return X.reset_index(drop=True)


def merge_emb(idx, emb_full, expected_dim, prefix):
    flat_cols = [c for c in emb_full.columns if c != 'ItemID']
    assert len(flat_cols) == expected_dim
    merged = df.iloc[idx][['ItemID']].astype({'ItemID': 'int64'}).reset_index(drop=True).merge(
        emb_full, on='ItemID', how='left'
    )
    arr = merged[flat_cols].fillna(0).values.astype('float32')
    missing = (merged[flat_cols].isna().all(axis=1)).mean()  # was 0 because of fillna; recompute via merged check
    return arr, float(merged[flat_cols[0]].isna().mean())


def run_multimodal(name, emb_full, expected_dim, prefix, n_components=25, save_to=None):
    print(f'\n========== {name} ==========')
    Xt = prep_tabular(train_idx); Xv = prep_tabular(val_idx); Xs = prep_tabular(test_idx)
    et, miss_t = merge_emb(train_idx, emb_full, expected_dim, prefix)
    ev, miss_v = merge_emb(val_idx,   emb_full, expected_dim, prefix)
    es, miss_s = merge_emb(test_idx,  emb_full, expected_dim, prefix)
    print(f'missing-emb rows: train={miss_t*100:.2f}%, val={miss_v*100:.2f}%, test={miss_s*100:.2f}%')

    pca = PCA(n_components=n_components, random_state=42)
    pt = pca.fit_transform(et); pv = pca.transform(ev); ps = pca.transform(es)
    evr = float(pca.explained_variance_ratio_.sum())
    print(f'PCA-{n_components} explained variance ratio = {evr:.4f}')

    pca_cols = [f'{prefix}_pca_{i}' for i in range(n_components)]
    Xt = pd.concat([Xt, pd.DataFrame(pt, columns=pca_cols)], axis=1)
    Xv = pd.concat([Xv, pd.DataFrame(pv, columns=pca_cols)], axis=1)
    Xs = pd.concat([Xs, pd.DataFrame(ps, columns=pca_cols)], axis=1)

    model = CatBoostClassifier(**CATBOOST_BASE)
    t0 = time.time()
    model.fit(Xt, y_train, cat_features=MY_CATS, eval_set=(Xv, y_val), use_best_model=True)
    dt = time.time() - t0
    proba = model.predict_proba(Xs)[:, 1].astype('float32')
    if save_to:
        np.save(save_to, proba)
    metrics = evaluate(y_test, proba, name)
    metrics.update({'best_iteration': int(model.best_iteration_),
                    'fit_seconds': round(dt, 1),
                    'pca_explained_variance': evr,
                    'missing_emb_test': miss_s})
    print(f'best_iter={metrics["best_iteration"]}, fit={dt:.1f}s')
    return proba, metrics, (pt, pv, ps)

In [ ]:
# ---- E5_team: CLIP + PCA-25 + my tabular ---------------------------------
proba_e5, m_e5, pca_clip = run_multimodal(
    'E5_team', clip_norm, expected_dim=512, prefix='clip',
    save_to=f'{PROBA}/test_proba_e5_team.npy',
)

In [ ]:
# ---- E7_team: e5-small text + PCA-25 + my tabular ------------------------
proba_e7, m_e7, pca_text = run_multimodal(
    'E7_team', text_emb, expected_dim=384, prefix='t',
    save_to=f'{PROBA}/test_proba_e7_team.npy',
)

In [ ]:
# ---- (Optional) Fusion: CLIP-PCA-25 + text-PCA-25 + my tabular -----------
# Skip this cell if you want to save Colab time. ~+5 min.
RUN_FUSION = True
if RUN_FUSION:
    Xt = prep_tabular(train_idx); Xv = prep_tabular(val_idx); Xs = prep_tabular(test_idx)
    pt_c, pv_c, ps_c = pca_clip
    pt_t, pv_t, ps_t = pca_text
    clip_cols = [f'clip_pca_{i}' for i in range(pt_c.shape[1])]
    text_cols = [f't_pca_{i}'    for i in range(pt_t.shape[1])]
    Xt = pd.concat([Xt, pd.DataFrame(pt_c, columns=clip_cols), pd.DataFrame(pt_t, columns=text_cols)], axis=1)
    Xv = pd.concat([Xv, pd.DataFrame(pv_c, columns=clip_cols), pd.DataFrame(pv_t, columns=text_cols)], axis=1)
    Xs = pd.concat([Xs, pd.DataFrame(ps_c, columns=clip_cols), pd.DataFrame(ps_t, columns=text_cols)], axis=1)
    print(f'Xt fusion shape: {Xt.shape}')

    model = CatBoostClassifier(**CATBOOST_BASE)
    t0 = time.time()
    model.fit(Xt, y_train, cat_features=MY_CATS, eval_set=(Xv, y_val), use_best_model=True)
    dt = time.time() - t0
    proba_fusion = model.predict_proba(Xs)[:, 1].astype('float32')
    np.save(f'{PROBA}/test_proba_fusion_team.npy', proba_fusion)
    m_fusion = evaluate(y_test, proba_fusion, 'Fusion_team')
    m_fusion.update({'best_iteration': int(model.best_iteration_), 'fit_seconds': round(dt, 1)})
    print(f'best_iter={m_fusion["best_iteration"]}, fit={dt:.1f}s')
else:
    m_fusion = None

In [ ]:
# ---- Save summary + zip artifacts for download ---------------------------
summary = {
    'catboost_base': {k: v for k, v in CATBOOST_BASE.items() if k != 'verbose'},
    'E5_team': m_e5,
    'E7_team': m_e7,
    'Fusion_team': m_fusion,
}
with open(f'{TR}/results/multimodal_team_summary.json', 'w') as f:
    json.dump(summary, f, indent=2, default=float)

import shutil
out_zip = '/content/team_runs_multimodal_outputs.zip'
with __import__('zipfile').ZipFile(out_zip, 'w') as z:
    for fname in ['test_proba_e5_team.npy', 'test_proba_e7_team.npy', 'test_proba_fusion_team.npy']:
        full = f'{PROBA}/{fname}'
        if os.path.exists(full):
            z.write(full, arcname=f'proba/{fname}')
    if os.path.exists(E5_PATH):
        z.write(E5_PATH, arcname='embeddings/text_e5_small.parquet')
    z.write(f'{TR}/results/multimodal_team_summary.json', arcname='results/multimodal_team_summary.json')
print(f'zip ready: {out_zip}')

from google.colab import files
files.download(out_zip)